### **Extract Atlas**

In [1]:
# Extract fMRI time series

import os
import re
import zipfile

input_root = r'D:\ukbiobank'
output_base = r'D:\organized_fMRI'

atlas_config = {
    '31016': ('fMRI.Glasser.csv.gz', 'Glasser'),
    '31019': ('fMRI.Tian_Subcortex_S4_3T.csv.gz', 'TianS4')
}

os.makedirs(output_base, exist_ok=True)

# Get list of already processed subject IDs (from folder names)
existing_subjects = set()
if os.path.exists(output_base):
    for item in os.listdir(output_base):
        if item.startswith('sub-') and os.path.isdir(os.path.join(output_base, item)):
            try:
                eid = int(item.replace('sub-', ''))
                existing_subjects.add(eid)
            except ValueError:
                continue  # skip malformed names

print(f"📁 Found {len(existing_subjects)} already processed subjects. Skipping them.")

def is_valid_zip(filepath):
    if not os.path.isfile(filepath) or os.path.getsize(filepath) == 0:
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            z.testzip()
        return True
    except:
        return False

for root, dirs, files in os.walk(input_root):
    for file in files:
        if not file.endswith('.zip'):
            continue

        match = re.match(r'(\d+)_(\d+)_(\d+)_(\d+)\.zip', file)
        if not match:
            continue

        eid, field_id, run_id, version = match.groups()
        eid_int = int(eid)

        # 🔁 SKIP if already extracted
        if eid_int in existing_subjects:
            continue

        if field_id not in atlas_config:
            continue

        zip_path = os.path.join(root, file)
        if not is_valid_zip(zip_path):
            continue

        internal_filename, atlas_name = atlas_config[field_id]

        try:
            with zipfile.ZipFile(zip_path, 'r') as z:
                if internal_filename not in z.namelist():
                    continue

                subject_dir = os.path.join(output_base, f"sub-{eid}")
                os.makedirs(subject_dir, exist_ok=True)

                out_name = f"sub-{eid}_run-{run_id}_{atlas_name}.csv.gz"
                out_path = os.path.join(subject_dir, out_name)

                # ⚠️ Only extract if file doesn't already exist (extra safety)
                if os.path.exists(out_path):
                    continue

                with z.open(internal_filename) as src, open(out_path, 'wb') as dst:
                    dst.write(src.read())

                print(f"✅ Extracted: {out_name}")

        except Exception as e:
            print(f"[ERROR] {zip_path}: {e}")

print("\n🎉 Extraction complete (skipped already processed subjects)!")

📁 Found 0 already processed subjects. Skipping them.

🎉 Extraction complete (skipped already processed subjects)!


### **Merge Atlas**

In [2]:
# Combine fMRI

# Libraries
import os
import pandas as pd
from tqdm import tqdm

fMRI_base = r'D:\organized_fMRI'
output_combined = r'D:\fMRI_combined'
os.makedirs(output_combined, exist_ok=True)

subjects = [d for d in os.listdir(fMRI_base) if d.startswith('sub-')]

for sub_dir in tqdm(subjects, desc="Merging subjects"):
    eid = sub_dir.replace('sub-', '')
    sub_path = os.path.join(fMRI_base, sub_dir)

    glasser_files = [f for f in os.listdir(sub_path) if 'Glasser' in f and 'run-2' in f]
    tian_files = [f for f in os.listdir(sub_path) if 'TianS4' in f and 'run-2' in f]

    if not glasser_files or not tian_files:
        continue

    g_file = os.path.join(sub_path, glasser_files[0])
    t_file = os.path.join(sub_path, tian_files[0])

    try:
        # Load and transpose — use first column as ROI names
        g_df = pd.read_csv(g_file, compression='gzip')
        t_df = pd.read_csv(t_file, compression='gzip')

        g = g_df.set_index(g_df.columns[0]).T
        t = t_df.set_index(t_df.columns[0]).T

        # ✅ Critical: within-subject time alignment
        if len(g) != len(t) or not g.index.equals(t.index):
            tqdm.write(f"[WARN] Time mismatch in {eid}, skipping.")
            continue

        combined = pd.concat([g, t], axis=1)

        out_file = os.path.join(output_combined, f"sub-{eid}_run-2_fMRI_combined.csv.gz")
        if os.path.exists(out_file):
            continue  # already done

        combined.to_csv(out_file, compression='gzip')

    except Exception as e:
        tqdm.write(f"[ERROR] {eid}: {e}")

print("\n🎉 Merging complete!")

Merging subjects: 0it [00:00, ?it/s]


🎉 Merging complete!


### **Explore the Data**

In [2]:
# inspect_fMRI_shapes.py
import os
import pandas as pd

input_dir = r'D:\fMRI_combined'
files = [f for f in os.listdir(input_dir) if f.endswith('_fMRI_combined.csv.gz')]
files = sorted(files)

# Inspect first 5 files
for fname in files[:5]:
    fpath = os.path.join(input_dir, fname)
    df = pd.read_csv(fpath, compression='gzip', index_col=0)
    print(f"File: {fname}")
    print(f"  Shape: {df.shape}")
    print(f"  First 3 timepoints: {df.index[:3].tolist()}")
    print(f"  Last 3 timepoints:  {df.index[-3:].tolist()}")
    print(f"  Regions: {len(df.columns)} (e.g., {df.columns[:3].tolist()} ... {df.columns[-3:].tolist()})")
    print("-" * 60)

File: sub-1000097_run-2_fMRI_combined.csv.gz
  Shape: (490, 414)
  First 3 timepoints: ['timepoint_0', 'timepoint_1', 'timepoint_2']
  Last 3 timepoints:  ['timepoint_487', 'timepoint_488', 'timepoint_489']
  Regions: 414 (e.g., ['R_V1_ROI', 'R_MST_ROI', 'R_V6_ROI'] ... ['NAc-core-lh', 'pGP-lh', 'aGP-lh'])
------------------------------------------------------------
File: sub-1000276_run-2_fMRI_combined.csv.gz
  Shape: (490, 414)
  First 3 timepoints: ['timepoint_0', 'timepoint_1', 'timepoint_2']
  Last 3 timepoints:  ['timepoint_487', 'timepoint_488', 'timepoint_489']
  Regions: 414 (e.g., ['R_V1_ROI', 'R_MST_ROI', 'R_V6_ROI'] ... ['NAc-core-lh', 'pGP-lh', 'aGP-lh'])
------------------------------------------------------------
File: sub-1000333_run-2_fMRI_combined.csv.gz
  Shape: (490, 414)
  First 3 timepoints: ['timepoint_0', 'timepoint_1', 'timepoint_2']
  Last 3 timepoints:  ['timepoint_487', 'timepoint_488', 'timepoint_489']
  Regions: 414 (e.g., ['R_V1_ROI', 'R_MST_ROI', 'R_V6_R

### **Structure the Data**

In [3]:
# structure_to_numpy_chunks.py
import os
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
from pandas.errors import EmptyDataError

input_dir = r'D:\fMRI_combined'
output_dir = r'D:\fMRI_numpy_chunks'
os.makedirs(output_dir, exist_ok=True)

files = sorted([f for f in os.listdir(input_dir) if f.endswith('_fMRI_combined.csv.gz')])
if not files:
    raise ValueError("No files found!")

# Get reference from first valid file
df0 = None
for f in files:
    try:
        df0 = pd.read_csv(os.path.join(input_dir, f), compression='gzip', index_col=0)
        if not df0.empty:
            break
    except:
        continue
if df0 is None:
    raise ValueError("No valid file for reference!")

region_names = df0.columns.tolist()
timepoints = df0.index.tolist()
T, R = len(timepoints), len(region_names)

# Save reference metadata once
metadata_ref = {
    'region_names': region_names,
    'timepoints': timepoints,
    'T': T,
    'R': R
}
with open(os.path.join(output_dir, 'reference_metadata.pkl'), 'wb') as f:
    pickle.dump(metadata_ref, f)

# Process in chunks
CHUNK_SIZE = 2000  # ~3–4 GB per chunk (2000 × 490 × 414 × 4 bytes ≈ 1.6 GB float32)
for chunk_start in range(0, len(files), CHUNK_SIZE):
    chunk_files = files[chunk_start:chunk_start + CHUNK_SIZE]
    chunk_data = []
    chunk_subjects = []

    print(f"\nProcessing chunk {chunk_start // CHUNK_SIZE + 1} ({len(chunk_files)} subjects)...")

    for fname in tqdm(chunk_files, desc=f"Chunk {chunk_start // CHUNK_SIZE + 1}"):
        eid = fname.split('_')[0].replace('sub-', '')
        fpath = os.path.join(input_dir, fname)

        try:
            df = pd.read_csv(fpath, compression='gzip', index_col=0)
            if df.empty:
                continue
        except Exception:
            continue

        # Align to reference
        df_aligned = df.reindex(index=timepoints, columns=region_names, fill_value=np.nan)
        chunk_data.append(df_aligned.values.astype(np.float32))
        chunk_subjects.append(eid)

    if not chunk_data:
        continue

    # Stack and save
    chunk_array = np.stack(chunk_data)  # shape: (N_chunk, T, R)
    chunk_id = chunk_start // CHUNK_SIZE
    np.save(os.path.join(output_dir, f'fMRI_chunk_{chunk_id:03d}.npy'), chunk_array)
    with open(os.path.join(output_dir, f'subjects_chunk_{chunk_id:03d}.pkl'), 'wb') as f:
        pickle.dump(chunk_subjects, f)

print("✅ All chunks saved!")


Processing chunk 1 (2000 subjects)...


Chunk 1: 100%|██████████| 2000/2000 [06:23<00:00,  5.21it/s]



Processing chunk 2 (2000 subjects)...


Chunk 2: 100%|██████████| 2000/2000 [06:05<00:00,  5.47it/s]



Processing chunk 3 (2000 subjects)...


...


Processing chunk 8 (2000 subjects)...


Chunk 8: 100%|██████████| 2000/2000 [06:45<00:00,  4.94it/s]



Processing chunk 9 (383 subjects)...


Chunk 9: 100%|██████████| 383/383 [01:10<00:00,  5.40it/s]


✅ All chunks saved!


> **data:** np.ndarray         
shape = (N_subjects, T_timepoints, R_regions)
> 

> **subject_ids:** list[str]   
['5200118', '5200235', ...]
> 

> **region_names:** list[str]  
['R_V1_ROI', ..., 'HIP-head-m1-rh', ...]
> 

> **timepoints:** list[str]    
['timepoint_0', ..., 'timepoint_489'] 

In [1]:
# merge_chunks.py
import os
import numpy as np
import pickle
from tqdm import tqdm

chunk_dir = r'C:\Users\yucca\Documents\Code\UBB\fMRI_numpy_chunks'
output_dir = r'C:\Users\yucca\Documents\Code\UBB\fMRI_numpy'
os.makedirs(output_dir, exist_ok=True)

# Load reference metadata
with open(os.path.join(chunk_dir, 'reference_metadata.pkl'), 'rb') as f:
    ref = pickle.load(f)

# Find all chunks
chunk_files = sorted([f for f in os.listdir(chunk_dir) if f.startswith('fMRI_chunk_') and f.endswith('.npy')])
subject_files = sorted([f for f in os.listdir(chunk_dir) if f.startswith('subjects_chunk_') and f.endswith('.pkl')])

if len(chunk_files) != len(subject_files):
    raise ValueError("Mismatch between chunk and subject files!")

print(f"🔍 Found {len(chunk_files)} chunks. Merging...")

all_data = []
all_subjects = []

# Load chunks with progress bar
for cf, sf in tqdm(zip(chunk_files, subject_files), total=len(chunk_files), desc="Loading chunks"):
    try:
        data = np.load(os.path.join(chunk_dir, cf))
        with open(os.path.join(chunk_dir, sf), 'rb') as f_sub:
            subjects = pickle.load(f_sub)
        all_data.append(data)
        all_subjects.extend(subjects)
    except Exception as e:
        print(f"\n⚠️ Skipping {cf}: {e}")

if not all_data:
    raise RuntimeError("No valid chunks loaded!")

# Concatenate
print("🧠 Concatenating arrays...")
final_data = np.concatenate(all_data, axis=0)
print(f"✅ Final shape: {final_data.shape}")

# Save as COMPRESSED .npz (avoids disk full errors)
output_file = os.path.join(output_dir, 'fMRI_data.npz')
print("💾 Saving compressed array...")
np.savez_compressed(output_file, data=final_data)

# Save metadata
metadata = {
    'subject_ids': all_subjects,
    'region_names': ref['region_names'],
    'timepoints': ref['timepoints'],
    'shape': final_data.shape,
    'dtype': str(final_data.dtype)
}
with open(os.path.join(output_dir, 'metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print(f"\n🎉 Done! Saved to:\n   {output_file}")
print(f"   Subject count: {len(all_subjects)}")
print(f"   Shape: {final_data.shape}")

🔍 Found 9 chunks. Merging...


Loading chunks: 100%|██████████| 9/9 [01:15<00:00,  8.40s/it]


🧠 Concatenating arrays...
✅ Final shape: (16382, 490, 414)
💾 Saving compressed array...

🎉 Done! Saved to:
   C:\Users\yucca\Documents\Code\UBB\fMRI_numpy\fMRI_data.npz
   Subject count: 16382
   Shape: (16382, 490, 414)
